In [ ]:
import Oceananigans as oc
using Oceananigans.Units

using CairoMakie
using CUDA
using Printf
using Random
using SeawaterPolynomials
using SeawaterPolynomials.TEOS10: TEOS10EquationOfState



Random.seed!(1969) # for reproducible results

In [ ]:
#########CLOCK & ARCHITECHTURE############
clock = oc.Clock(; time = 0.0)
architecture = oc.GPU()

############GRID DEFINITION###############

# Number of cells
Cx = 100
Cz = 60

# Domain extent
Lx = 100  # m
Lz = 60   # m

# Toggle vertical stretching
use_stretching = true

if use_stretching
    refinement = 1.05
    stretching = 12
    h(k)  = (k - 1) / Cz
    ζ₀(k) = 1 + (h(k) - 1) / refinement
    Σ(k)  = (1 - exp(-stretching * h(k))) / (1 - exp(-stretching))
    z_interfaces(k) = Lz * (ζ₀(k) * Σ(k) - 1)
else
    z_interfaces = (-Lz, 0)
end

# 2D grid: actually a pseudo 3D grid
grid = oc.RectilinearGrid(architecture,
                       size = (Cx,10, Cz),
                       y = (0,10),
                       x = (0, Lx),
                       z = z_interfaces,
                       topology = (oc.Periodic, oc.Periodic, oc.Bounded))

In [ ]:

# wind stress parameters
u₁₀ = 10  # m s⁻¹, average wind velocity 10 meters above the ocean
cᴰ = 2e-3 # dimensionless drag coefficient
ρₐ = 1.2  # kg m⁻³, approximate average density of air at sea-level
ρₒ = 1e3
τx = - ρₐ / ρₒ * cᴰ * u₁₀ * abs(u₁₀) # m² s⁻²

# evaporation rate (changes salinity only in this case)
evaporation_rate = 1e-3 / hour # m s⁻¹

# heat fluxes parameters
Q = 0*200   # W m⁻², surface _heat_ flux
cᴾ = 3991 # J K⁻¹ kg⁻¹, typical heat capacity for seawater

# temperature gradients (also used in initial conditions)
dTdz = 0.01 # K m⁻¹   # set up temperature gradient (water column and bottom)


@inline Jˢ(x,y, t, S, evaporation_rate) = - evaporation_rate * S # [salinity unit] m s⁻¹


# -- set up details on boundary conditions representation in the model -- #
evaporation_bc = oc.FluxBoundaryCondition(Jˢ,
                        field_dependencies=:S,
                        parameters=evaporation_rate)


Jᵀ = Q / (ρₒ * cᴾ) # K m s⁻¹, surface _temperature_ flux

# different types of boundary conditions formulations are available
temperature_top    =  oc.FluxBoundaryCondition(Jᵀ)        
temperature_bottom =  oc.GradientBoundaryCondition(dTdz)
#temperature_top = ValueBoundaryCondition(T_surface_bc)



# -- set up details on wind stress parameterization -- $
# u_bcs = oc.FieldBoundaryConditions(top = oc.FluxBoundaryCondition(τx))
@inline bottom_drag(x,y, t, u) = - 1e-3 * u^2  # quadratic drag


# -- set up the boundaries of the model + formulations -- #
u_bcs = oc.FieldBoundaryConditions(top = oc.FluxBoundaryCondition(τx),
                                # bottom = ValueBoundaryCondition(0))
                                # bottom = oc.FluxBoundaryCondition(bottom_drag, field_dependencies=:u)
                                )


S_bcs = oc.FieldBoundaryConditions(top=evaporation_bc)

T_bcs = oc.FieldBoundaryConditions(
    top    = temperature_top,
    bottom = temperature_bottom
)



# -- organize everything in one single structure that will be passed to the model-- #
boundary_conditions = ( T = T_bcs,
                        S = S_bcs,
                        u = u_bcs)


In [ ]:
# Buoyancy formulation: converts T and S into density anomalies using the
# TEOS-10 equation of state, which is the international standard for seawater thermodynamics.
# This enables buoyancy-driven flow (convection, stratification) in the model.
ρₒ = 1026
buoyancy = oc.SeawaterBuoyancy(equation_of_state = TEOS10EquationOfState(reference_density=ρₒ))

In [ ]:
###############DEFINE MODEL################
# model configurations
model = oc.NonhydrostaticModel(grid;
                            clock = clock,
                            advection = oc.WENO(order=5),
                            closure = oc.DynamicSmagorinsky(),
                            # closure = oc.ScalarDiffusivity(ν=1e-2, κ=1e-2),
                            # closure = oc.AnisotropicMinimumDissipation(),
                            boundary_conditions = boundary_conditions,
                            coriolis = oc.FPlane(f=1e-4),
                            # biogeochemistry = biogeochemistry,
                            tracers = (:T, :S),
                            buoyancy = buoyancy
                            )      


In [ ]:

########## Initial conditions for 2D #############

Ξ(z) = randn() * z / model.grid.Lz * (1 + z / model.grid.Lz) # noise
Ξ2(x, y,  z) = randn()

# Temperature initial condition: a stable density gradient with random noise superposed.
Tᵢ(x, y, z) = 20 + 1e-7* Ξ2(x, y, z) + dTdz * z + dTdz * model.grid.Lz * 2e-6 * Ξ(z)

@inline Sᵢ(x, y, z) = 34 + 1e-6* Ξ2(x, y, z)

uᵢ(x, y, z) = sqrt(abs(τx)) * 1e-3 * Ξ(z)

# #Units in mmol N/m3
oc.set!(model, T = Tᵢ, S = Sᵢ,u=uᵢ)

# Define output writers:
T = model.tracers.T
S = model.tracers.S
u = model.velocities.u
v = model.velocities.v
w = model.velocities.w

In [ ]:
# T_init = Array(oc.interior(model.tracers.T, :, 1, :))
# zf = Array(oc.znodes(grid, oc.Face()))
# x  = Array(oc.xnodes(grid,oc.Center()))
# z  = Array(oc.znodes(grid,oc.Center()))

# Δz = diff(zf)

# fig = Figure(size = (800, 400))

# ax_T = Axis(fig[1, 3]; xlabel = "x (m)", ylabel = "z (m)", title = "Initial Temperature")
# ax_z = Axis(fig[1, 1]; xlabel = "vertical spacing", ylabel = "z (m)", title = "Vertical grid")
# ax_z2 = Axis(fig[1, 2]; xlabel = "vertical spacing", ylabel = "z (m)", title = "Vertical grid")

# hm = heatmap!(ax_T, x, z, T_init; colormap = :thermal)
# Colorbar(fig[1, 4], hm; label = "°C")

# scatter!(ax_z, Δz, zf[1:end-1]; markersize = 8)
# ax_z.xlabel = "Δz (m)"
# scatter!(ax_z2, 1:length(zf)-1, zf[1:end-1]; markersize = 8, color = :red)
# ax_z2.xlabel = "grid index"

# fig

In [ ]:

# 1. Define the stop time for the simulation
stop_time = 6.0hours


# 2. Create the simulation with a starting Δt and the stop_time
simulation = oc.Simulation(model, Δt=20, stop_time=stop_time)

# 3. Define a progress message callback
progress(sim) = @info @printf("Iter: %d, time: %s, Δt: %s, max|w|: %.2e m/s, wall time: %s",
                                oc.iteration(sim),
                                oc.prettytime(sim),
                                oc.prettytime(sim.Δt),
                                maximum(abs, sim.model.velocities.w),
                                oc.prettytime(sim.run_wall_time))

simulation.callbacks[:progress] = oc.Callback(progress, oc.IterationInterval(100))

# 4. Update the output writer to save results once per day
simulation.output_writers[:jld2] = oc.JLD2Writer(model,
                                (; T, S, u,v,w),
                                filename = "ocean_convection3.jld2",
                                schedule = oc.TimeInterval(1minute),
                                overwrite_existing = true)

oc.conjure_time_step_wizard!(simulation, cfl=0.7, max_Δt=20)


In [ ]:
simulation.model.clock.time = 0.  # comment this to restart (othrerwise it will start from 0)
oc.run!(simulation)               # running the simulation

In [ ]:

# plotting structure

eos = TEOS10EquationOfState()
simulation.model.clock.time = 0.  # uncomment this to restart from 0
filepath = "ocean_convection3.jld2"
Tt = oc.FieldTimeSeries(filepath, "T")
St = oc.FieldTimeSeries(filepath, "S")
w = oc.FieldTimeSeries(filepath, "w")
u = oc.FieldTimeSeries(filepath, "u")
v = oc.FieldTimeSeries(filepath, "v")

times = Tt.times
n = Observable(length(times))

Tₙ = @lift oc.interior(Tt[$n], :, 1, :)
Sₙ = @lift oc.interior(St[$n], :, 1, :)
ρₙ = @lift SeawaterPolynomials.ρ.(oc.interior(Tt[$n], :, 1, :),
                                   oc.interior(St[$n], :, 1, :),
                                   0.0,
                                   Ref(eos))
wₙ = @lift oc.interior(w[$n], :, 1, :)
uₙ = @lift oc.interior(u[$n], :, 1, :)
vₙ = @lift oc.interior(v[$n], :, 1, :)

grid = Tt.grid
x = oc.xnodes(grid, oc.Center())
z = oc.znodes(grid, oc.Center())
zf = oc.znodes(grid, oc.Face())

title = @lift @sprintf("t = %s", oc.prettytime(times[$n]))

axis_kwargs = (xlabel = "x (m)",
               ylabel = "z (m)",
               limits = ((0, grid.Lx), (-grid.Lz, 0)))

Tlims = (17., 22)
ulims = (-0.2,0.2)
vlims = (-0.2,0.2)


fig = Figure(size = (1200, 800), figure_padding = 5)

fig[0, 1:6] = Label(fig, title, fontsize=16, tellwidth=false)

ax_T = Axis(fig[1, 1]; title = "Temperature", axis_kwargs...)
ax_S = Axis(fig[1, 3]; title = "Salinity",    axis_kwargs...)
ax_ρ = Axis(fig[1, 5]; title = "Density",     axis_kwargs...)
ax_u = Axis(fig[2, 1]; title = "u",           axis_kwargs...)
ax_w = Axis(fig[2, 3]; title = "w",           axis_kwargs...)
ax_v = Axis(fig[2, 5]; title = "v", axis_kwargs...)

hm_T = heatmap!(ax_T, x,  z,  Tₙ; colormap = :thermal)
Colorbar(fig[1, 2], hm_T; label = "°C")

hm_S = heatmap!(ax_S, x, z,  Sₙ; colormap = :haline)
Colorbar(fig[1, 4], hm_S; label = "g / kg")

hm_ρ = heatmap!(ax_ρ, x, z,  ρₙ; colormap = :dense)
Colorbar(fig[1, 6], hm_ρ; label = "kg / m³")

hm_u = heatmap!(ax_u, x, z,  uₙ; colormap = :balance, colorrange=ulims)
Colorbar(fig[2, 2], hm_u; label = "m/s")

hm_v = heatmap!(ax_v, x, z, vₙ; colormap = :balance, colorrange = vlims)
Colorbar(fig[2, 6], hm_v; label = "m/s")

hm_w = heatmap!(ax_w, x, zf, wₙ; colormap = :balance)
Colorbar(fig[2, 4], hm_w; label = "m/s")

fig


In [ ]:
# creating a movie

filename = "ocean_convection3"

intro = searchsortedfirst(times, 10minutes)
frames = intro:length(times)

@info "Making a motion picture..."
resize_to_layout!(fig)
CairoMakie.record(fig, filename * ".mp4", frames, framerate=8) do i
    n[] = i
end